# Session Memory — LangGraph Agent Tutorial

LangGraph has no built-in "session" concept the way it has a checkpointer for threads — but its
cross-thread **`Store`** (LangGraph's mechanism for memory that spans threads) supports a native
**TTL** (time-to-live), which is exactly the primitive session memory needs: state that lives
for *one visit*, not forever.

This notebook builds a LangGraph agent whose node reads/writes session-scoped preferences
through `SqliteStore` configured with a TTL, and shows precisely how — and how imperfectly —
LangGraph enforces that expiry.

In [1]:
# ============ IMPORTS ============
import os
import sys
import sqlite3
import time

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.sqlite import SqliteStore
from langgraph.store.base import TTLConfig
from langgraph.config import get_store

sys.path.append(os.path.abspath("../../.."))
from helpers import get_llm

from dotenv import load_dotenv
load_dotenv()

print("Imports OK")

Imports OK


In [2]:
# ============ LLM INITIALIZATION ============
llm = get_llm()

LLM initialized: system.ai.gemma-3-12b (via databricks_gateway)


## 1. A Store With a TTL

`SqliteStore` is LangGraph's long-term (cross-thread) memory backend. Passing a `TTLConfig`
turns it into a session store: every item written gets an expiry, and (if `refresh_on_read` is
set) a read slides the expiry forward — the same "touch on activity" behavior a web session
cookie has.

`default_ttl` is in **minutes**. We use a short one here (`0.05` ≈ 3 seconds) purely so the
expiry demo doesn't require actually waiting 30 minutes.

In [3]:
# ============ SESSION STORE (TTL-BACKED) ============
DB_PATH = "session_memory.db"
# isolation_level=None hands transaction control to the store itself -- required for
# SqliteStore's own BEGIN/COMMIT handling to work correctly.
conn = sqlite3.connect(DB_PATH, check_same_thread=False, isolation_level=None)

session_store = SqliteStore(
    conn,
    ttl=TTLConfig(
        default_ttl=0.05,        # ~3 seconds -- short for this demo; use e.g. 30 (minutes) in production
        refresh_on_read=False,   # keep the expiry demo deterministic; flip to True for sliding sessions
        omit_expired=True,
        sweep_interval_minutes=None,  # we sweep manually below instead of a background thread
    ),
)
session_store.setup()
print("session store ready")

session store ready


## 2. A Session-Aware Node

The node reads a session preference (e.g. "reply in French for this visit") from the store
*before* calling the LLM, using `session_id` — not `thread_id` — as the store key. This is what
separates session memory from short-term memory: it survives a brand-new thread as long as the
`session_id` is the same and the session hasn't expired.

In [4]:
# ============ SESSION-AWARE AGENT NODE ============
def agent_node(state: MessagesState, config) -> dict:
    session_id = config["configurable"]["session_id"]
    pref = session_store.get(("session", session_id), "reply_style")
    style_note = pref.value["instruction"] if pref else "Respond normally."

    system = SystemMessage(f"Session instruction for this visit only: {style_note}")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

# No checkpointer needed to demonstrate session memory in isolation -- each call is a
# fresh thread on purpose, to prove the preference survives via the STORE, not the checkpointer.
session_agent = builder.compile(store=session_store)
print("Session-aware agent compiled.")

Session-aware agent compiled.


## 3. Setting a Session Preference and Using It From a New Thread

A user sets a mid-visit preference. A brand-new `thread_id` under the **same** `session_id`
still picks it up — session memory is keyed by the visit, not the conversation thread.

In [5]:
# ============ DEMO: SESSION PREFERENCE SURVIVES A NEW THREAD ============
SESSION_ID = "session-alice-visit-1"

session_store.put(
    ("session", SESSION_ID),
    "reply_style",
    {"instruction": "Reply only in French for the rest of this visit."},
)

cfg_thread_1 = {"configurable": {"session_id": SESSION_ID, "thread_id": "thread-1"}}
cfg_thread_2 = {"configurable": {"session_id": SESSION_ID, "thread_id": "thread-2"}}  # different thread!

out1 = session_agent.invoke({"messages": [HumanMessage("Say hello.")]}, cfg_thread_1)
print("thread-1 reply:", out1["messages"][-1].content)

out2 = session_agent.invoke({"messages": [HumanMessage("Say hello again.")]}, cfg_thread_2)
print("thread-2 reply (new thread, SAME session):", out2["messages"][-1].content)

thread-1 reply: Bonjour !

thread-2 reply (new thread, SAME session): Bonjour !


## 4. A Different Session Sees Nothing

Session scope means a new `session_id` — a new visit, even for the same person — starts with no
preference.

In [6]:
# ============ DEMO: A NEW SESSION IS ISOLATED ============
cfg_new_session = {"configurable": {"session_id": "session-alice-visit-2", "thread_id": "thread-3"}}
out3 = session_agent.invoke({"messages": [HumanMessage("Say hello.")]}, cfg_new_session)
print("new session reply (no French instruction carried over):", out3["messages"][-1].content)

new session reply (no French instruction carried over): Hello! 😊 



(Just acknowledging the session instruction - back to normal responses now!)


## 5. Expiry — What `get()` Actually Does (and Doesn't Do)

This is the gotcha worth seeing with your own eyes rather than taking on faith: `store.get()`
does **not** reliably hide an expired item on its own, even with `omit_expired=True`, until a
**sweep** actually runs. Expiry is enforced by the sweeper, not by every read.

In [7]:
# ============ EXPIRY DEMO ============
print("immediately after write:", session_store.get(("session", SESSION_ID), "reply_style"))

time.sleep(4)  # longer than our 0.05-minute (~3s) TTL

still_there = session_store.get(("session", SESSION_ID), "reply_style")
print("after the TTL has elapsed, before a sweep:", still_there)
# If this still prints the item, that's not a bug in this notebook -- it's how the store
# actually behaves: expiry is enforced by the SWEEPER, not implicitly by every get().

swept = session_store.sweep_ttl()
print(f"\nswept {swept} expired row(s)")

after_sweep = session_store.get(("session", SESSION_ID), "reply_style")
print("after the sweep:", after_sweep)

immediately after write: Item(namespace=['session', 'session-alice-visit-1'], key='reply_style', value={'instruction': 'Reply only in French for the rest of this visit.'}, created_at='2026-08-25T06:27:15', updated_at='2026-08-25T06:27:15')
after the TTL has elapsed, before a sweep: Item(namespace=['session', 'session-alice-visit-1'], key='reply_style', value={'instruction': 'Reply only in French for the rest of this visit.'}, created_at='2026-08-25T06:27:15', updated_at='2026-08-25T06:27:15')

swept 1 expired row(s)
after the sweep: None


## 6. Reading From an Expired Session in the Agent

Once swept, the session-aware node falls back to its default behavior — exactly like a session
cookie that's gone.

In [8]:
# ============ DEMO: AGENT BEHAVIOR AFTER EXPIRY ============
out4 = session_agent.invoke({"messages": [HumanMessage("Say hello.")]}, cfg_thread_1)
print("thread-1 reply after session expiry:", out4["messages"][-1].content)

thread-1 reply after session expiry: Hello! 😊 



(Just acknowledging the session instruction - back to normal responses now!)


## Gotchas

- **`omit_expired=True` is not a read-time guarantee by itself.** As shown above, `get()` can
  still return a technically-expired item until `sweep_ttl()` (or the background
  `start_ttl_sweeper()`) actually deletes it. Don't assume expiry is enforced on every read —
  either sweep on a schedule, or explicitly check `updated_at`/age yourself if a stale read for
  a few seconds/minutes is unacceptable.
- **`default_ttl` is in minutes, not seconds** — an easy off-by-unit bug (`ttl=30` meaning 30
  *minutes*, not 30 seconds, surprises people coming from Redis-style TTL-in-seconds APIs).
- **`refresh_on_read` changes the whole session model.** `True` gives sliding expiry (any
  activity extends the visit); `False` gives a fixed-length session regardless of activity.
  Picking the wrong one silently changes your security/UX assumptions.
- **Session vs. thread key confusion.** This notebook deliberately used different `thread_id`s
  under the same `session_id` to prove the distinction — using `thread_id` as the store's
  namespace key by mistake would make "session" memory collapse back into short-term memory.
- **Session state must not silently become long-term memory.** Nothing stops you from also
  writing the same key into a `("semantic", user_id)` namespace — but that must be a deliberate,
  separate write (see `05_Long_Term_Semantic_Memory_SQLite.ipynb`), not an accident of a session
  entry that nobody ever sweeps.
- **`isolation_level=None` on the sqlite3 connection is required.** `SqliteStore` manages its own
  `BEGIN`/`COMMIT`; the default DBAPI auto-transaction behavior conflicts with it and raises
  `cannot start a transaction within a transaction`.

## Key Takeaways

- Session memory in LangGraph = a `Store` (here, `SqliteStore`) configured with `TTLConfig`,
  keyed by `session_id` rather than `thread_id` — it can span multiple threads.
- Expiry is only real once something sweeps — verify that assumption rather than trusting it.
- This is a different LangGraph primitive from the checkpointer used for short-term memory, and
  a narrower-scoped one than the plain `Store` usage for long-term memory in the next notebooks.
